# Lab 01 · Anatomy of a response
**~20 minutes · costs about $0.01 · Domain 2 (33.1%), Domain 6 Output Handling**

Four exam facts, proven by running them:

1. `system` is a top-level parameter, not a message role
2. `content` is a **list of blocks**, not a string
3. `stop_reason` arrives on a **successful 200** and is not an error
4. `usage` has more fields than you think

In [ ]:
import os, anthropic
client = anthropic.Anthropic()          # reads ANTHROPIC_API_KEY
MODEL  = "claude-sonnet-4-6"            # verify against lab 00 output
CHEAP  = "claude-haiku-4-5"             # for high-volume steps
print("sdk", anthropic.__version__)

## 1 · The shape of the object

Print the whole thing once. Most people never do this.

In [ ]:
r = client.messages.create(
    model=MODEL, max_tokens=300,
    system="You are terse.",
    messages=[{"role":"user","content":"Name three primary colours."}],
)
print(r.model_dump_json(indent=2))

## 2 · Iterate blocks by type

`r.content[0].text` works right up until a `thinking` or `tool_use` block lands
in position zero. Then it throws in production at 2am.

In [ ]:
def render(resp):
    for i, b in enumerate(resp.content):
        print(f"[{i}] type={b.type}")
        if b.type == "text":
            print("    ", b.text[:120].replace("\n"," "))
    return resp

render(r);

## 3 · Force a truncation and read stop_reason

This returns **HTTP 200**. It is not an error. Your code must branch on it.

In [ ]:
t = client.messages.create(
    model=MODEL, max_tokens=16,                       # deliberately tiny
    messages=[{"role":"user","content":"Explain the TCP handshake in detail."}],
)
print("stop_reason:", t.stop_reason)
print("text so far:", repr(t.content[0].text))
print()
print("TRUNCATED, NOT MALFORMED. The rest was never generated.")
print("Sending this back to the model for 'repair' asks it to invent the missing part.")

## 4 · Prove that `system` is not a role

Expected: a **400 invalid_request_error**. Read the message it returns.

In [ ]:
import anthropic
try:
    client.messages.create(model=MODEL, max_tokens=64, messages=[
        {"role":"system","content":"You are terse."},
        {"role":"user","content":"hi"},
    ])
    print("no error - unexpected")
except anthropic.APIStatusError as e:
    print("status:", e.status_code)
    print(e.message[:300])

## 5 · Every field of usage

Log these separately in production or your cost model is quietly wrong.

In [ ]:
u = r.usage
for f in ["input_tokens","output_tokens","cache_creation_input_tokens","cache_read_input_tokens"]:
    print(f"{f:<30}", getattr(u, f, None))

---
### Checkpoint
Without looking, say out loud:
- the five `stop_reason` values and why none is an error
- why `content[0].text` is a latent bug
- what you do when `stop_reason == "max_tokens"` and the JSON is broken